# SEMOVI shp -> SQl

Objective: Process, clean, standarize and create a geodf for each mode of public transport. To upload to our databse in sql.

Final result: Schema: SEMOVI, 
For each transport system
table: transport_system_stops, transport_system_route

Lastly a unified stop table

We only care about: Transport system, stop (geom)

## SQL setup

In [1456]:
# Cell 0 — imports, connection, helpers, PostGIS enable

import os, re
from pathlib import Path
import pandas as pd
import geopandas as gpd
import fiona
from sqlalchemy import create_engine, text

# ---- Edit here or set as env vars in your shell ----
os.environ.setdefault("PGHOST", "localhost")
os.environ.setdefault("PGPORT", "5432")
os.environ.setdefault("PGDATABASE", "dataton")
os.environ.setdefault("PGUSER", "postgres")
os.environ.setdefault("PGPASSWORD", "5548")
# ----------------------------------------------------

def engine():
    return create_engine(
        f"postgresql+psycopg2://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}"
        f"@{os.environ['PGHOST']}:{os.environ['PGPORT']}/{os.environ['PGDATABASE']}",
        future=True
    )

def ensure_schema(schema: str):
    with engine().begin() as con:
        con.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema}"'))

def to_snake(s: str) -> str:
    s = re.sub(r"[^\w]+", "_", s)
    s = re.sub(r"_{2,}", "_", s)
    return s.strip("_").lower() or "col"

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols, seen = [], set()
    for c in df.columns:
        base = to_snake(str(c))
        name = base
        i = 2
        while name in seen:
            name = f"{base}_{i}"; i += 1
        seen.add(name); cols.append(name)
    df = df.copy(); df.columns = cols
    return df

def write_tabular(df: pd.DataFrame, schema: str, table: str, if_exists="replace"):
    df = normalize_columns(df)
    df.to_sql(table, con=engine(), schema=schema, if_exists=if_exists, index=False, chunksize=5000)

def write_spatial(gdf: gpd.GeoDataFrame, schema: str, table: str, if_exists="replace"):
    # normalize non-geometry columns
    non_geom = [c for c in gdf.columns if c != gdf.geometry.name]
    gdf = gdf.rename(columns={c: to_snake(c) for c in non_geom})
    # rename geometry to 'geom'
    if gdf.geometry.name != "geom":
        gdf = gdf.rename_geometry("geom")
    # set/reproject to EPSG:4326 for consistent SRID
    if gdf.crs is None:
        gdf.set_crs(4326, inplace=True)
    else:
        gdf = gdf.to_crs(4326)
    # write
    gdf.to_postgis(
        name=table, con=engine(), schema=schema,
        if_exists=if_exists, index=False, chunksize=5000
    )
    # gist index
    with engine().begin() as con:
        con.execute(text(
            f'CREATE INDEX IF NOT EXISTS "idx_{schema}_{table}_geom" '
            f'ON "{schema}"."{table}" USING GIST ("geom")'
        ))

# Enable PostGIS (run once per database)
with engine().begin() as con:
    con.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))

# Resolve project root (your notebook lives inside a subfolder; the root has INEGI/, MGE/, OSM/)
def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if all((p/d).is_dir() for d in ("INEGI", "MGE", "OSM")):
            return p
    return start

ROOT = find_root(Path.cwd().resolve())
ROOT

WindowsPath('C:/Users/Edu/OneDrive/1.Data/0.Base')

## Cleaning

### Utils

In [1357]:
from shapely.ops import transform
from unidecode import unidecode


In [1358]:
#3D point to 2d
def to_2d(x, y, z):
    return (x, y)

#unidecode, no spaces, lowercase, underscores
def clean_string_columns(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(
                lambda x: unidecode(str(x)).lower().replace(" ", "_") if isinstance(x, str) else x
            )
    return df

def rows_with_nan_geom(df):
    return df[df['geom'].isna()]

def get_duplicates(df, subset=None):
    """
    Returns duplicate rows in a DataFrame or GeoDataFrame.
    :param df: DataFrame or GeoDataFrame
    :param subset: Columns to consider for identifying duplicates (default: all columns)
    :return: DataFrame of duplicate rows
    """
    dup_mask = df.duplicated(subset=subset, keep=False)
    return df[dup_mask]


### Concesionados

#### Paths

In [1359]:
paradasConce = gpd.read_file("Data/Unzipped/concesionado_shp/concesionado_paradas.shp")
rutasConce = gpd.read_file("Data/Unzipped/concesionado_shp/concesionado_rutas.shp")

#### Stops

In [1360]:
paradasConce

,SISTEMA,EMPRESA,ORIG_DEST,NOMEN_OD,RUTA,UBICACION,NOMENCL,CORREDOR,NOMEN_CORR,geometry
0,CORREDORES CONCESIONADOS,COTAXOMIL,Tasqueña - Madreselva,A,Canal de Miramontes,Calzada Taxqueña,25A,XOCHIMILCO,25,POINT Z (-99.1376 19.34146 0)
1,CORREDORES CONCESIONADOS,COTAXOMIL,Tasqueña - Madreselva,A,Canal de Miramontes,Cerro de San Antonio,25A,XOCHIMILCO,25,POINT Z (-99.13742 19.33936 0)
2,CORREDORES CONCESIONADOS,COTAXOMIL,Tasqueña - Madreselva,A,Canal de Miramontes,Gálvez,25A,XOCHIMILCO,25,POINT Z (-99.13716 19.33661 0)
3,CORREDORES CONCESIONADOS,COTAXOMIL,Tasqueña - Madreselva,A,Canal de Miramontes,Av. de El Parque,25A,XOCHIMILCO,25,POINT Z (-99.13712 19.33364 0)
4,CORREDORES CONCESIONADOS,COTAXOMIL,Tasqueña - Madreselva,A,Canal de Miramontes,La Virgen,25A,XOCHIMILCO,25,POINT Z (-99.13708 19.33065 0)
...,...,...,...,...,...,...,...,...,...,...
13079,CORREDORES CONCESIONADOS,CONGESA,None,D,Metro la Raza Linea 5 - Juanacatlán,Circuito Interior -Sandalo,20D,METRO AEROPUERTO - METRO TACUBAYA,20,POINT Z (-99.14822 19.46321 2224.04)
13080,CORREDORES CONCESIONADOS,CONGESA,None,D,Metro la Raza Linea 5 - Juanacatlán,Circuito Interior - Hospital de la Raza,20D,METRO AEROPUERTO - METRO TACUBAYA,20,POINT Z (-99.14498 19.46416 2227.86)
13081,CORREDORES CONCESIONADOS,CONGESA,None,D,Metro la Raza Linea 5 - Juanacatlán,Insurgentes - Clave,20D,METRO AEROPUERTO - METRO TACUBAYA,20,POINT Z (-99.14074 19.46617 0)
13082,CORREDORES CONCESIONADOS,CONGESA,None,D,Metro la Raza Linea 5 - Juanacatlán,Insurgentes - CETRAM La Raza,20D,METRO AEROPUERTO - METRO TACUBAYA,20,POINT Z (-99.13876 19.46818 2220.02)


In [1361]:
paradasConceP = paradasConce[["SISTEMA", "RUTA", "UBICACION", "geometry"]].copy()

In [1362]:
paradasConceP = paradasConceP.rename(columns={"SISTEMA": "tipo_transporte", "RUTA": "linea", "UBICACION": "parada", "geometry": "geom"})
paradasConceP['tipo_transporte']='CC'
paradasConceP['geom'] = paradasConceP['geom'].apply(lambda geom: transform(to_2d, geom))
paradasConceP = clean_string_columns(paradasConceP)
paradasConceP.sort_values(by=["linea"], inplace=True, ignore_index=True)


In [1363]:
rows_with_nan_geom(paradasConceP)

,tipo_transporte,linea,parada,geom


In [1364]:
get_duplicates(paradasConceP, subset=["linea","parada"])

,tipo_transporte,linea,parada,geom
20,cc,1deg_de_mayo_-_chapultepec,manuel_avila_camacho_y_legaria,POINT (-99.21647 19.44241)
24,cc,1deg_de_mayo_-_chapultepec,base_manuel_avila_camacho_y_las_torres,POINT (-99.22489 19.46556)
29,cc,1deg_de_mayo_-_chapultepec,base_manuel_avila_camacho_y_las_torres,POINT (-99.22449 19.46592)
44,cc,1deg_de_mayo_-_chapultepec,manuel_avila_camacho_y_legaria,POINT (-99.21539 19.44171)
51,cc,1deg_de_mayo_-_chapultepec,base_cetram_chapultepec,POINT (-99.17648 19.42158)
...,...,...,...,...
13036,cc,xochimilco_-_izazaga_por_miramontes,av._guadalupe_i._ramirez_y_la_noria,POINT (-99.12684 19.26861)
13044,cc,xochimilco_-_izazaga_por_miramontes,av._guadalupe_i._ramirez_y_rubellon,POINT (-99.11943 19.26702)
13048,cc,xochimilco_-_izazaga_por_miramontes,base_pino_y_benito_juarez,POINT (-99.10397 19.26442)
13052,cc,xochimilco_-_izazaga_por_miramontes,base_pino_y_benito_juarez,POINT (-99.10397 19.26442)


I dont know what to do with this duplicates, their geoms arent duplicated

#### Route


In [1365]:
rutasConce.head(5)

,ORIG_DEST,RUTA,SISTEMA,CORREDOR,NOMEN_COR,NOMEN_OD,EMPRESA,NOMENCL,geometry
0,Tasqueña - Madreselva,Tasqueña - Madreselva,CORREDORES CONCESIONADOS,XOCHIMILCO,25,A,COTAXOMIL,25A,"LINESTRING Z (-99.13843 19.34401 0, -99.13806 ..."
1,Madreselva - Tasqueña,Madreselva - Tasqueña,CORREDORES CONCESIONADOS,XOCHIMILCO,25,A,COTAXOMIL,25A,"LINESTRING Z (-99.09943 19.25146 0, -99.09948 ..."
2,Metro Tasqueña - Galeana,Metro Tasqueña - Galeana,CORREDORES CONCESIONADOS,XOCHIMILCO,25,B,COTAXOMIL,25B,"LINESTRING Z (-99.13782 19.34367 0, -99.13759 ..."
3,Galeana - Metro Tasqueña,Galeana - Metro Tasqueña,CORREDORES CONCESIONADOS,XOCHIMILCO,25,B,COTAXOMIL,25B,"LINESTRING Z (-99.09675 19.25053 0, -99.09652 ..."
4,Tasqueña - Milpa Alta,Tasqueña - Milpa Alta,CORREDORES CONCESIONADOS,XOCHIMILCO,25,C,COTAXOMIL,25C,"LINESTRING Z (-99.14012 19.34228 0, -99.14092 ..."


In [1366]:
rutasConceP = rutasConce[["SISTEMA", "ORIG_DEST", "RUTA", "geometry"]].copy()


In [1367]:
rutasConceP.rename(columns={"SISTEMA": "tipo_transporte", "ORIG_DEST": "origen_destino", "RUTA": "ruta", "geometry": "geom"}, inplace=True)
rutasConceP['tipo_transporte']='CC'
rutasConceP['geom'] = rutasConceP['geom'].apply(lambda geom: transform(to_2d, geom))
rutasConceP = clean_string_columns(rutasConceP)

In [1368]:
rows_with_nan_geom(rutasConceP)

,tipo_transporte,origen_destino,ruta,geom


In [1369]:
get_duplicates(rutasConceP, subset=["tipo_transporte", "origen_destino", "ruta"])

,tipo_transporte,origen_destino,ruta,geom


#### Finals

In [1370]:
paradasConceP.head(5)

,tipo_transporte,linea,parada,geom
0,cc,10_de_abril_-_centro_de_azcapotzalco,calle_miguel_lerdo_de_tejada_-_calle_santo_dom...,POINT (-99.19295 19.48321)
1,cc,10_de_abril_-_centro_de_azcapotzalco,calle_soberania_estatal_-_calle_gral._lazaro_c...,POINT (-99.21961 19.49183)
2,cc,10_de_abril_-_centro_de_azcapotzalco,c._miguel_velazquez_mancilla_-_calz._de_san_ag...,POINT (-99.21679 19.49291)
3,cc,10_de_abril_-_centro_de_azcapotzalco,calle_venustiano_carranza_-_calle_felipe_angeles,POINT (-99.21228 19.49144)
4,cc,10_de_abril_-_centro_de_azcapotzalco,calle_francisco_sarabia_-_calle_manuel_salazar,POINT (-99.2079 19.48962)


In [1371]:
rutasConceP.head(5)

,tipo_transporte,origen_destino,ruta,geom
0,cc,tasquena_-_madreselva,tasquena_-_madreselva,"LINESTRING (-99.13843 19.34401, -99.13806 19.3..."
1,cc,madreselva_-_tasquena,madreselva_-_tasquena,"LINESTRING (-99.09943 19.25146, -99.09948 19.2..."
2,cc,metro_tasquena_-_galeana,metro_tasquena_-_galeana,"LINESTRING (-99.13782 19.34367, -99.13759 19.3..."
3,cc,galeana_-_metro_tasquena,galeana_-_metro_tasquena,"LINESTRING (-99.09675 19.25053, -99.09652 19.2..."
4,cc,tasquena_-_milpa_alta,tasquena_-_milpa_alta,"LINESTRING (-99.14012 19.34228, -99.14092 19.3..."


### Metrobus

#### Paths

In [1372]:
paradasMB = gpd.read_file("Data/Unzipped/mb_shp/Metrobus_estaciones.shp")
rutasMB= gpd.read_file("Data/Unzipped/mb_shp/Metrobus_lineas.shp")

#### Stops

In [1373]:
paradasMB

,SISTEMA,NOMBRE,LINEA,EST,CVE_EOD17,TIPO,ALCALDIAS,AÑO,CVE_EST,geometry
0,Metrobús,París,07,None,None,Servicio temporal,Cuauhtémoc,2023.0,None,POINT Z (-99.15591 19.43239 0)
1,Metrobús,Reforma,07,None,None,Servicio temporal,Cuauhtémoc,2023.0,None,POINT Z (-99.15854 19.43111 0)
2,Metrobús,Hamburgo,07,None,None,Servicio temporal,Cuauhtémoc,2023.0,None,POINT Z (-99.1628 19.42917 0)
3,Metrobús,El Ahuehuete,07,None,None,Servicio temporal,Cuauhtémoc,2023.0,None,POINT Z (-99.16506 19.42807 0)
4,Metrobús,El Ángel,07,None,None,Servicio temporal,Cuauhtémoc,2023.0,None,POINT Z (-99.16866 19.42642 0)
...,...,...,...,...,...,...,...,...,...,...
319,Metrobús,Auditorio,07,30,None,Intermedia,Miguel Hidalgo,2018.0,MB0730,POINT Z (-99.1942 19.42609 0)
320,Metrobús,Campo Marte,07,31,None,Terminal,Miguel Hidalgo,2018.0,MB0731,POINT Z (-99.19936 19.42689 0)
321,Metrobús,Indios Verdes,01,01,None,None,Gustavo A. Madero,NaN,MB0101,POINT Z (-99.11971 19.49685 0)
322,Metrobús,Buenavista II,01,47,None,Terminal,Cuauhtémoc,NaN,MB0147,POINT Z (-99.15206 19.44639 0)


In [1374]:
paradasMBP = paradasMB[["SISTEMA", "LINEA","NOMBRE", "geometry"]].copy()

In [1375]:
paradasMBP.rename(columns={"SISTEMA": "tipo_transporte","NOMBRE": "parada", "LINEA": "linea", "geometry": "geom"}, inplace=True)
paradasMBP['tipo_transporte']='MB'
paradasMBP['geom'] = paradasMBP['geom'].apply(lambda geom: transform(to_2d, geom))
paradasMBP = clean_string_columns(paradasMBP)
paradasMBP.sort_values(by=["linea"], inplace=True, ignore_index=True)

In [1376]:
rows_with_nan_geom(paradasMBP)

,tipo_transporte,linea,parada,geom


In [1377]:
get_duplicates(paradasMBP, subset=["parada", "linea"])

,tipo_transporte,linea,parada,geom
122,mb,04,defensoria_publica,POINT (-99.15063 19.43311)
123,mb,04,defensoria_publica,POINT (-99.15101 19.43336)
124,mb,04,amajac,POINT (-99.15337 19.43501)
125,mb,04,amajac,POINT (-99.15353 19.43416)
126,mb,04,plaza_de_la_republica,POINT (-99.15398 19.43713)
...,...,...,...,...
312,mb,07,el_ahuehuete,POINT (-99.16483 19.4285)
313,mb,07,hamburgo,POINT (-99.1615 19.43005)
314,mb,07,reforma,POINT (-99.15871 19.43143)
315,mb,07,paris,POINT (-99.15596 19.43266)


No dropeare, son las distintas subidas y bajadas en una misma est.

In [1378]:
paradasMBP

,tipo_transporte,linea,parada,geom
0,mb,01,felix_cuevas,POINT (-99.1788 19.37425)
1,mb,01,colonia_del_valle,POINT (-99.17509 19.38569)
2,mb,01,cd._de_los_deportes,POINT (-99.17614 19.38245)
3,mb,01,parque_hundido,POINT (-99.17707 19.37959)
4,mb,01,rio_churubusco,POINT (-99.18057 19.36878)
...,...,...,...,...
319,mb,07,gustavo_a._madero,POINT (-99.11393 19.48355)
320,mb,07,av._talisman,POINT (-99.12108 19.47868)
321,mb,07,necaxa,POINT (-99.12232 19.47538)
322,mb,07,robles_dominguez,POINT (-99.12518 19.46776)


#### Routes

In [1379]:
rutasMB

,SISTEMA,RUTA,TRAMO,LINEA,geometry
0,Metrobús,Tacubaya - París,Tacubaya - París,07,"MULTILINESTRING Z ((-99.18437 19.40767 0, -99...."
1,Metrobús,Buenavista - San Lázaro Ruta Sur,None,04,"LINESTRING Z (-99.15227 19.44494 0, -99.15254 ..."
2,Metrobús,Buenavista - San Lázaro Ruta Sur,None,04,"LINESTRING Z (-99.13379 19.42926 0, -99.13259 ..."
3,Metrobús,Buenavista - San Lázaro Ruta Sur,None,04,"LINESTRING Z (-99.11988 19.42553 0, -99.11375 ..."
4,Metrobús,Buenavista - San Lázaro Ruta Sur,None,04,"LINESTRING Z (-99.15324 19.43536 0, -99.15303 ..."
...,...,...,...,...,...
59,Metrobús,Hospital Infantil La Villa - Campo Marte,Garrido - Campo Marte,07,"LINESTRING Z (-99.11935 19.48325 0, -99.1199 1..."
60,Metrobús,Juárez - Río Frío,Río Frío - Juárez,02 y 03,"LINESTRING Z (-99.14802 19.4313 0, -99.14788 1..."
61,Metrobús,Río Frío - Juárez,Río Frío - Juárez,02 y 03,"LINESTRING Z (-99.07428 19.38763 0, -99.07469 ..."
62,Metrobús,Sta. Cruz Atoyac - Indios Verdes,Sta. Cruz Atoyac - Indios Verdes,03,"LINESTRING Z (-99.12114 19.49323 0, -99.12028 ..."


In [1380]:
rutasMBP = rutasMB[["SISTEMA", "LINEA", "RUTA", "geometry"]].copy()

In [1381]:
rutasMBP.rename(columns={"SISTEMA": "tipo_transporte", "LINEA": "linea", "RUTA": "ruta", "geometry": "geom"}, inplace=True)
rutasMBP['tipo_transporte']='MB'
rutasMBP['geom'] = rutasMBP['geom'].apply(lambda geom: transform(to_2d, geom))
rutasMBP = clean_string_columns(rutasMBP)
rutasMBP.sort_values(by=["linea"], inplace=True, ignore_index=True)

In [1382]:
rows_with_nan_geom(rutasMBP)

,tipo_transporte,linea,ruta,geom


In [1383]:
get_duplicates(rutasMBP, subset=["ruta", "linea"])

,tipo_transporte,linea,ruta,geom
4,mb,01_y_02,dr._galvez_(linea_1)_a_rojo_gomez_(linea_2),"LINESTRING (-99.19009 19.34074, -99.18898 19.3..."
5,mb,01_y_02,dr._galvez_(linea_1)_a_rojo_gomez_(linea_2),"LINESTRING (-99.07635 19.38453, -99.08897 19.3..."
6,mb,02,tepalcates_-_etiopia,"LINESTRING (-99.06582 19.38347, -99.06649 19.3..."
7,mb,02,tepalcates_-_etiopia,"LINESTRING (-99.05985 19.38903, -99.06087 19.3..."
8,mb,02,tepalcates_-_etiopia,"LINESTRING (-99.05985 19.38903, -99.05974 19.3..."
9,mb,02,tepalcates_-_tacubaya,"LINESTRING (-99.18351 19.40759, -99.18412 19.4..."
10,mb,02,tepalcates_-_etiopia,"LINESTRING (-99.07637 19.38451, -99.07704 19.3..."
11,mb,02,tepalcates_-_tacubaya,"LINESTRING (-99.07637 19.38451, -99.0759 19.38..."
12,mb,02,tepalcates_-_tacubaya,"LINESTRING (-99.06582 19.38347, -99.06649 19.3..."
13,mb,02,tepalcates_-_tacubaya,"LINESTRING (-99.05985 19.38903, -99.06087 19.3..."


In [1384]:
#check qgis to drop if needed
#rutasMBP = rutasMBP.drop_duplicates(subset=["linea", "ruta"], ignore_index=True)

#### Finals

In [1385]:
paradasMBP.head(5)

,tipo_transporte,linea,parada,geom
0,mb,01,felix_cuevas,POINT (-99.1788 19.37425)
1,mb,01,colonia_del_valle,POINT (-99.17509 19.38569)
2,mb,01,cd._de_los_deportes,POINT (-99.17614 19.38245)
3,mb,01,parque_hundido,POINT (-99.17707 19.37959)
4,mb,01,rio_churubusco,POINT (-99.18057 19.36878)


In [1386]:
rutasMBP.head(5)

,tipo_transporte,linea,ruta,geom
0,mb,01,indios_verdes_-_insurgentes,"LINESTRING (-99.11972 19.49682, -99.11961 19.4..."
1,mb,01,indios_verdes_-_el_caminero,"LINESTRING (-99.11972 19.49682, -99.11961 19.4..."
2,mb,01,indios_verdes_-_doctor_galvez,"LINESTRING (-99.11972 19.49682, -99.11961 19.4..."
3,mb,01,buenavista_-_el_caminero,"LINESTRING (-99.15207 19.44641, -99.15256 19.4..."
4,mb,01_y_02,dr._galvez_(linea_1)_a_rojo_gomez_(linea_2),"LINESTRING (-99.19009 19.34074, -99.18898 19.3..."


### RTP

#### Paths

In [1387]:
paradasRTP = gpd.read_file("Data/Unzipped/rtp_shp/RTP_paradas.shp")
rutasRTP= gpd.read_file("Data/Unzipped/rtp_shp/RTP_lineas.shp")

#### Stops

In [1388]:
paradasRTP.head(15)

,RUTA,MODULO,SENTIDO,ORIG_DEST,CORREDOR,INSTERSECC,SISTEMA,geometry
0,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Ixtlahuaca,Red de Transporte de Pasajeros,POINT Z (-99.27207 19.33469 0)
1,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Potrero de Tepito,Red de Transporte de Pasajeros,POINT Z (-99.26808 19.33713 0)
2,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Calz. Las Águilas,Red de Transporte de Pasajeros,POINT Z (-99.26418 19.33937 0)
3,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Estrellas Colgate,Red de Transporte de Pasajeros,POINT Z (-99.2595 19.34062 0)
4,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Parque Axomiatla,Red de Transporte de Pasajeros,POINT Z (-99.25432 19.34202 0)
5,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Plaza Axomiatla,Red de Transporte de Pasajeros,POINT Z (-99.25211 19.34092 0)
6,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Walmart Express,Red de Transporte de Pasajeros,POINT Z (-99.24827 19.3406 0)
7,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Gavilanes,Red de Transporte de Pasajeros,POINT Z (-99.24457 19.34254 0)
8,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Supervía Poniente,Red de Transporte de Pasajeros,POINT Z (-99.24166 19.34363 0)
9,115-A,1,SN,San Bartolo Ameyalco (Cafeteros) - Metro Chapu...,None,Las Águilas,Red de Transporte de Pasajeros,POINT Z (-99.23768 19.34555 0)


In [1389]:
paradasRTPP = paradasRTP[["SISTEMA", "RUTA","INSTERSECC", "geometry"]].copy()

In [1390]:
paradasRTPP.rename(columns={"SISTEMA": "tipo_transporte", "INSTERSECC": "parada", "RUTA": "linea", "geometry": "geom"}, inplace=True)
paradasRTPP['tipo_transporte']='RTP'
paradasRTPP['geom'] = paradasRTPP['geom'].apply(lambda geom: transform(to_2d, geom))
paradasRTPP = clean_string_columns(paradasRTPP)
paradasRTPP.sort_values(by=["linea"], inplace=True, ignore_index=True)

In [1391]:
get_duplicates(paradasRTPP, subset=["parada", "linea"])

,tipo_transporte,linea,parada,geom
3,rtp,1-d,primavera,POINT (-99.01786 19.35028)
16,rtp,1-d,la_viga,POINT (-99.12147 19.35779)
17,rtp,1-d,sur_73,POINT (-99.13448 19.35909)
22,rtp,1-d,cetram_santa_martha_anden_a,POINT (-98.99514 19.35996)
26,rtp,1-d,cuauhtemoc,POINT (-99.00395 19.35682)
...,...,...,...,...
8118,rtp,9-d,tec_de_monterrey,POINT (-99.26105 19.36109)
8123,rtp,9-d,samara,POINT (-99.25997 19.36838)
8128,rtp,9-d,ibero,POINT (-99.264 19.36769)
8130,rtp,9-d,centro_comercial_santa_fe,POINT (-99.27236 19.3637)


In [1392]:
rows_with_nan_geom(paradasRTPP)

,tipo_transporte,linea,parada,geom


#### Routes

In [1393]:
rutasRTP

,NOMBRE,SISTEMA,RUTA,MODULO,ORIGEN,DESTINO,ORDINARIO,ATENEA,EXPRESO,ECOBUS,EXPRESO_DI,NOCHEBUS,ESTATUS,geometry
0,San Bartolo - Chapultepec,Red de Transporte de Pasajeros,115-A,1,San Bartolo,Chapultepec,NO,NO,SI,NO,NO,NO,ACTIVA,"LINESTRING Z (-99.2748 19.33273 0, -99.27466 1..."
1,Chapultepec - San Bartolo,Red de Transporte de Pasajeros,115-A,1,Chapultepec,San Bartolo,NO,NO,SI,NO,NO,NO,ACTIVA,"LINESTRING Z (-99.1761 19.42182 0, -99.17634 1..."
2,Acoxpa - Santa Fe / UAM Cuajimalpa,Red de Transporte de Pasajeros,300-B,2,Acoxpa,Santa Fe,NO,NO,NO,NO,SI,NO,ACTIVA,"LINESTRING Z (-99.13836 19.29916 0, -99.13834 ..."
3,Santa Fe / UAM Cuajimalpa - Acoxpa,Red de Transporte de Pasajeros,300-B,2,Santa Fe,Acoxpa,NO,NO,NO,NO,SI,NO,ACTIVA,"LINESTRING Z (-99.2819 19.35326 0, -99.28172 1..."
4,Palmitas - Metro Constitución de 1917,Red de Transporte de Pasajeros,159,4,Palmitas,Constitución de 1917,SI,NO,NO,NO,NO,NO,ACTIVA,"LINESTRING Z (-99.02532 19.33138 0, -99.02529 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,Metro El Rosario - Metro Chapultepec,Red de Transporte de Pasajeros,59,7,Chapultepec,El Rosario,SI,NO,NO,NO,NO,NO,ACTIVA,"LINESTRING Z (-99.17587 19.42222 0, -99.17594 ..."
196,Metro El Rosario - Sullivan,Red de Transporte de Pasajeros,59-A,7,El Rosario,Sullivan,SI,SI,NO,NO,NO,NO,ACTIVA,"LINESTRING Z (-99.2001 19.50606 0, -99.20006 1..."
197,Metro El Rosario - Sullivan,Red de Transporte de Pasajeros,59-A,7,Sullivan,El Rosario,SI,SI,NO,NO,NO,NO,ACTIVA,"LINESTRING Z (-99.16057 19.43414 0, -99.16051 ..."
198,Santa Fe (Centro Comercial) - San Bartolo Amey...,Red de Transporte de Pasajeros,9-D,1,Santa Fe,San Bartolo,NO,NO,SI,NO,NO,NO,ACTIVA,"LINESTRING Z (-99.27968 19.36107 0, -99.27954 ..."


In [1394]:
rutasRTP = rutasRTP[rutasRTP["ESTATUS"]=='ACTIVA']

In [1395]:
rutasRTPP = rutasRTP[["SISTEMA", "NOMBRE", "RUTA", "geometry"]].copy()

In [1396]:
rutasRTPP.rename(columns={"SISTEMA": "tipo_transporte", "NOMBRE": "ruta", "RUTA": "identificador_ruta", "geometry": "geom"}, inplace=True)
rutasRTPP['tipo_transporte']='RTP'
rutasRTPP['geom'] = rutasRTPP['geom'].apply(lambda geom: transform(to_2d, geom))
rutasRTPP = clean_string_columns(rutasRTPP)
rutasRTPP = rutasRTPP.sort_values(by=["identificador_ruta"], ignore_index=True)

In [1397]:
get_duplicates(rutasRTPP, subset=["ruta", "identificador_ruta"])

,tipo_transporte,ruta,identificador_ruta,geom
0,rtp,metro_santa_marta_-_metro_mixcoac,1-d,"LINESTRING (-98.99528 19.36007, -98.99442 19.3..."
1,rtp,metro_santa_marta_-_metro_mixcoac,1-d,"LINESTRING (-99.18804 19.37577, -99.18786 19.3..."
2,rtp,col._lomas_de_cuautepec_-_metro_indios_verdes,101,"LINESTRING (-99.12095 19.49352, -99.12115 19.4..."
3,rtp,col._lomas_de_cuautepec_-_metro_indios_verdes,101,"LINESTRING (-99.13469 19.57094, -99.13474 19.5..."
4,rtp,ampliacion_malacates_-_la_villaferroplaza,101-a,"LINESTRING (-99.13086 19.57633, -99.13075 19.5..."
...,...,...,...,...
195,rtp,san_gregorio_atlapulco_-_metro_tasquena,81-a,"LINESTRING (-99.13913 19.34465, -99.13862 19.3..."
196,rtp,centro_comercial_santa_fe_-_puerta_grande,9-c,"LINESTRING (-99.24426 19.35378, -99.24423 19.3..."
197,rtp,centro_comercial_santa_fe_-_puerta_grande,9-c,"LINESTRING (-99.27968 19.36107, -99.27954 19.3..."
198,rtp,santa_fe_(centro_comercial)_-_san_bartolo_amey...,9-d,"LINESTRING (-99.27968 19.36107, -99.27954 19.3..."


diferente geom, no se si dropear

In [1398]:
rows_with_nan_geom(rutasRTPP)

,tipo_transporte,ruta,identificador_ruta,geom


#### Finals

In [1399]:
paradasRTPP

,tipo_transporte,linea,parada,geom
0,rtp,1-d,avena,POINT (-99.11882 19.35746)
1,rtp,1-d,zacatepec,POINT (-99.0088 19.35516)
2,rtp,1-d,octavio_senties_ii,POINT (-99.01216 19.35375)
3,rtp,1-d,primavera,POINT (-99.01786 19.35028)
4,rtp,1-d,las_palmas_(la_quebradora),POINT (-99.02026 19.34593)
...,...,...,...,...
8127,rtp,9-d,estacionamiento_ibero,POINT (-99.26283 19.36885)
8128,rtp,9-d,ibero,POINT (-99.264 19.36769)
8129,rtp,9-d,autopista,POINT (-99.26819 19.36557)
8130,rtp,9-d,centro_comercial_santa_fe,POINT (-99.27236 19.3637)


In [1400]:
rutasRTPP

,tipo_transporte,ruta,identificador_ruta,geom
0,rtp,metro_santa_marta_-_metro_mixcoac,1-d,"LINESTRING (-98.99528 19.36007, -98.99442 19.3..."
1,rtp,metro_santa_marta_-_metro_mixcoac,1-d,"LINESTRING (-99.18804 19.37577, -99.18786 19.3..."
2,rtp,col._lomas_de_cuautepec_-_metro_indios_verdes,101,"LINESTRING (-99.12095 19.49352, -99.12115 19.4..."
3,rtp,col._lomas_de_cuautepec_-_metro_indios_verdes,101,"LINESTRING (-99.13469 19.57094, -99.13474 19.5..."
4,rtp,ampliacion_malacates_-_la_villaferroplaza,101-a,"LINESTRING (-99.13086 19.57633, -99.13075 19.5..."
...,...,...,...,...
195,rtp,san_gregorio_atlapulco_-_metro_tasquena,81-a,"LINESTRING (-99.13913 19.34465, -99.13862 19.3..."
196,rtp,centro_comercial_santa_fe_-_puerta_grande,9-c,"LINESTRING (-99.24426 19.35378, -99.24423 19.3..."
197,rtp,centro_comercial_santa_fe_-_puerta_grande,9-c,"LINESTRING (-99.27968 19.36107, -99.27954 19.3..."
198,rtp,santa_fe_(centro_comercial)_-_san_bartolo_amey...,9-d,"LINESTRING (-99.27968 19.36107, -99.27954 19.3..."


### METRO

#### Paths

In [1401]:
paradasM = gpd.read_file("Data/Unzipped/stcmetro_shp/STC_Metro_estaciones_utm14n.shp")
rutasM = gpd.read_file("Data/Unzipped/stcmetro_shp/STC_Metro_lineas_utm14n.shp")

#### Stops

In [1402]:
paradasM

,SISTEMA,NOMBRE,LINEA,EST,CVE_EST,CVE_EOD17,TIPO,ALCALDIAS,AÑO,geometry
0,STC Metro,Pantitlán,01,01,STC0101,05014,Terminal / Transbordo,Venustiano Carranza,1984,POINT Z (-99.07474 19.41633 0)
1,STC Metro,Zaragoza,01,02,STC0102,05020,Intermedia,Venustiano Carranza,1969,POINT Z (-99.08229 19.41192 0)
2,STC Metro,Gomez Farías,01,03,STC0103,05007,Intermedia,Venustiano Carranza,1969,POINT Z (-99.09021 19.41648 0)
3,STC Metro,Boulevard Puerto Aéreo,01,04,STC0104,05003,Intermedia,Venustiano Carranza,1969,POINT Z (-99.09626 19.41994 0)
4,STC Metro,Balbuena,01,05,STC0105,05001,Intermedia,Venustiano Carranza,1969,POINT Z (-99.10277 19.42336 0)
...,...,...,...,...,...,...,...,...,...,...
190,STC Metro,Múzquiz,B,17,STCLB17,05185,Intermedia,Estado de México - Ecatepec,2000,POINT Z (-99.04207 19.50149 0)
191,STC Metro,Ecatepec,B,18,STCLB18,05179,Intermedia,Estado de México - Ecatepec,2000,POINT Z (-99.03607 19.51513 0)
192,STC Metro,Olímpica,B,19,STCLB19,05188,Intermedia,Estado de México - Ecatepec,2000,POINT Z (-99.03334 19.52134 0)
193,STC Metro,Plaza Aragón,B,20,STCLB20,05189,Intermedia,Estado de México - Ecatepec,2000,POINT Z (-99.03018 19.52848 0)


In [1403]:
paradasMP = paradasM[["SISTEMA","LINEA", "NOMBRE", "geometry"]].copy()

In [1404]:
paradasMP.rename(columns={"SISTEMA": "tipo_transporte","NOMBRE": "parada", "LINEA": "linea", "geometry": "geom"}, inplace=True)
paradasMP['tipo_transporte']='M'
paradasMP['geom'] = paradasMP['geom'].apply(lambda geom: transform(to_2d, geom))
paradasMP = clean_string_columns(paradasMP)
paradasMP.sort_values(by=["linea"], inplace=True, ignore_index=True)

In [1405]:
get_duplicates(paradasMP, subset=["parada", "linea"])

,tipo_transporte,linea,parada,geom


In [1406]:
rows_with_nan_geom(paradasMP)

,tipo_transporte,linea,parada,geom


#### Routes

In [1407]:
rutasM

,SISTEMA,LINEA,RUTA,geometry
0,STC Metro,1,Observatorio - Pantitlàn,"LINESTRING Z (478958.079 2144909.693 0, 479222..."
1,STC Metro,2,Cuatro Caminos - Tasqueña,"LINESTRING Z (477345.656 2151695.912 0, 477344..."
2,STC Metro,3,Indios Verdes - Universidad,"LINESTRING Z (487459.272 2155641.308 0, 487436..."
3,STC Metro,4,Martín Carrera - Santa Anita,"LINESTRING Z (487222.685 2145411.999 0, 487225..."
4,STC Metro,5,Politecnico - Pantitlán,"LINESTRING Z (484343.637 2156234.759 0, 484375..."
5,STC Metro,6,El Rosario - Martín Carrera,"LINESTRING Z (479038.409 2156677.397 0, 479009..."
6,STC Metro,7,El Rosario - Barranca del Muerto,"LINESTRING Z (479002.112 2156689.361 0, 478981..."
7,STC Metro,8,Garibaldi - Constitución de 1917,"LINESTRING Z (485409.03 2149962.292 0, 485363...."
8,STC Metro,9,Tacubaya - Pantitlán,"LINESTRING Z (492419.448 2146787.767 0, 492363..."
9,STC Metro,A,Pantitlán - La Paz,"LINESTRING Z (492449.095 2146781.819 0, 492410..."


In [1408]:
rutasMP = rutasM[["SISTEMA", "LINEA", "RUTA", "geometry"]].copy()

In [1409]:
rutasMP.rename(columns={"SISTEMA": "tipo_transporte","LINEA": "linea", "RUTA": "ruta", "geometry": "geom"}, inplace=True)
rutasMP['tipo_transporte']='M'
rutasMP['geom'] = rutasMP['geom'].apply(lambda geom: transform(to_2d, geom))
rutasMP = clean_string_columns(rutasMP)
rutasMP['linea'] = rutasMP['linea'].apply(lambda x: f"0{x}" if isinstance(x, str) and x.isdigit() and len(x) == 1 else x) #same format as stops
rutasMP.sort_values(by=["linea"], inplace=True, ignore_index=True)


In [1410]:
get_duplicates(rutasMP, subset=["linea", "ruta"])


,tipo_transporte,linea,ruta,geom


In [1411]:
rows_with_nan_geom(rutasMP)

,tipo_transporte,linea,ruta,geom


#### Finals

In [1412]:
paradasMP

,tipo_transporte,linea,parada,geom
0,m,01,pantitlan,POINT (-99.07474 19.41633)
1,m,01,observatorio,POINT (-99.2004 19.39828)
2,m,01,tacubaya,POINT (-99.18764 19.40251)
3,m,01,juanacatlan,POINT (-99.18198 19.41295)
4,m,01,chapultepec,POINT (-99.17642 19.42075)
...,...,...,...,...
190,m,b,garibaldi/lagunilla,POINT (-99.13787 19.44389)
191,m,b,guerrero,POINT (-99.14614 19.44514)
192,m,b,plaza_aragon,POINT (-99.03018 19.52848)
193,m,b,oceania,POINT (-99.08722 19.44569)


In [1413]:
rutasMP

,tipo_transporte,linea,ruta,geom
0,m,01,observatorio_-_pantitlan,"LINESTRING (478958.079 2144909.693, 479222.974..."
1,m,02,cuatro_caminos_-_tasquena,"LINESTRING (477345.656 2151695.912, 477344.608..."
2,m,03,indios_verdes_-_universidad,"LINESTRING (487459.272 2155641.308, 487436.281..."
3,m,04,martin_carrera_-_santa_anita,"LINESTRING (487222.685 2145411.999, 487225.748..."
4,m,05,politecnico_-_pantitlan,"LINESTRING (484343.637 2156234.759, 484375.744..."
5,m,06,el_rosario_-_martin_carrera,"LINESTRING (479038.409 2156677.397, 479009.369..."
6,m,07,el_rosario_-_barranca_del_muerto,"LINESTRING (479002.112 2156689.361, 478981.374..."
7,m,08,garibaldi_-_constitucion_de_1917,"LINESTRING (485409.03 2149962.292, 485363.164 ..."
8,m,09,tacubaya_-_pantitlan,"LINESTRING (492419.448 2146787.767, 492363.703..."
9,m,12,mixcoac_-_tlahuac,"LINESTRING (480391.136 2142409.146, 480561.434..."


### CABLEBUS

#### Paths

In [1414]:
paradasCB = gpd.read_file("Data/Unzipped/ste_shp/ste_cablebus_shp/STE_Cablebus_estaciones.shp")
rutasCB= gpd.read_file("Data/Unzipped/ste_shp/ste_cablebus_shp/STE_Cablebus_lineas.shp")

#### Stops

In [1415]:
paradasCB

,SISTEMA,NOMBRE,LINEA,EST,CVE_EST,TIPO,AÑO,Elevadores,Guia_tact,P_braile,ALCALDIAS,geometry
0,STE Cablebús,Indios Verdes,01,01,CB0101,Terminal,2021,Sí,Sí,No,Gustavo A. Madero,POINT Z (-99.12226 19.49544 0)
1,STE Cablebús,Ticomán,01,02,CB0102,Intermedia,2021,Sí,Sí,No,Gustavo A. Madero,POINT Z (-99.1338 19.51167 0)
2,STE Cablebús,La Pastora,01,03,CB0103,Intermedia,2021,Sí,Sí,No,Gustavo A. Madero,POINT Z (-99.1408 19.52735 0)
3,STE Cablebús,Campos Revolución,01,04,CB0104,Intermedia,2021,Sí,Sí,No,Gustavo A. Madero,POINT Z (-99.14196 19.54027 0)
4,STE Cablebús,Cuautepec,01,05,CB0105,Terminal,2021,Sí,Sí,No,Gustavo A. Madero,POINT Z (-99.13416 19.55795 0)
5,STE Cablebús,Tlalpexco,01,06,CB0106,Antena,2021,Sí,Sí,No,Gustavo A. Madero,POINT Z (-99.12643 19.54477 0)
6,STE Cablebús,Constitución de 1917,02,01,CB0201,Terminal,2021,Sí,Sí,Sí,Iztapalapa,POINT Z (-99.06251 19.34486 0)
7,STE Cablebús,Quetzacóatl,02,02,CB0202,Intermedia,2021,Sí,Sí,Sí,Iztapalapa,POINT Z (-99.04345 19.32783 0)
8,STE Cablebús,Las Torres Buenavista,02,03,CB0203,Intermedia,2021,Sí,Sí,Sí,Iztapalapa,POINT Z (-99.03083 19.33179 0)
9,STE Cablebús,Xalpa,02,04,CB0204,Intermedia,2021,Sí,Sí,Sí,Iztapalapa,POINT Z (-99.019 19.33617 0)


In [1416]:
paradasCBP = paradasCB[["SISTEMA", "LINEA","NOMBRE", "geometry"]].copy()

In [1417]:
paradasCBP.rename(columns={"SISTEMA": "tipo_transporte","NOMBRE": "parada", "LINEA": "linea", "geometry": "geom"}, inplace=True)
paradasCBP['tipo_transporte']='CB'
paradasCBP['geom'] = paradasCBP['geom'].apply(lambda geom: transform(to_2d, geom))
paradasCBP = clean_string_columns(paradasCBP)
paradasCBP.sort_values(by=["linea"], inplace=True, ignore_index=True)

In [1418]:
rows_with_nan_geom(paradasCBP)

,tipo_transporte,linea,parada,geom


In [1419]:
get_duplicates(paradasCBP, subset=["parada", "linea"])

,tipo_transporte,linea,parada,geom


#### Routes

In [1420]:
rutasCB

,SISTEMA,LINEA,RUTA,geometry
0,STE Cablebús,1,Indios Verdes - Cuautepec,"LINESTRING Z (-99.12226 19.49544 0, -99.1338 1..."
1,STE Cablebús,1,Antena Tlalpexco,"LINESTRING Z (-99.12643 19.54477 0, -99.14196 ..."
2,STE Cablebús,2,Constitución de 1917 - Santa Marta,"LINESTRING Z (-99.06251 19.34486 0, -99.04345 ..."
3,STE Cablebús,3,Los Pinos/Constituyentes - Vasco de Quiroga,"LINESTRING Z (-99.2318 19.38422 0, -99.22905 1..."


In [1421]:
rutasCBP = rutasCB[["SISTEMA", "LINEA", "RUTA", "geometry"]].copy()

In [1422]:
rutasCBP.rename(columns={"SISTEMA": "tipo_transporte", "LINEA": "linea", "RUTA": "ruta", "geometry": "geom"}, inplace=True)
rutasCBP['tipo_transporte']='CB'
rutasCBP['geom'] = rutasCBP['geom'].apply(lambda geom: transform(to_2d, geom))
rutasCBP = clean_string_columns(rutasCBP)
rutasCBP.sort_values(by=["linea"], inplace=True, ignore_index=True)

#### Finals

In [1423]:
paradasCBP

,tipo_transporte,linea,parada,geom
0,cb,01,indios_verdes,POINT (-99.12226 19.49544)
1,cb,01,ticoman,POINT (-99.1338 19.51167)
2,cb,01,la_pastora,POINT (-99.1408 19.52735)
3,cb,01,campos_revolucion,POINT (-99.14196 19.54027)
4,cb,01,cuautepec,POINT (-99.13416 19.55795)
5,cb,01,tlalpexco,POINT (-99.12643 19.54477)
6,cb,02,santa_marta,POINT (-98.99398 19.35952)
7,cb,02,san_miguel_teotongo,POINT (-98.98955 19.34353)
8,cb,02,lomas_de_la_estancia,POINT (-99.00811 19.34083)
9,cb,02,xalpa,POINT (-99.019 19.33617)


In [1424]:
rutasCBP

,tipo_transporte,linea,ruta,geom
0,cb,1,indios_verdes_-_cuautepec,"LINESTRING (-99.12226 19.49544, -99.1338 19.51..."
1,cb,1,antena_tlalpexco,"LINESTRING (-99.12643 19.54477, -99.14196 19.5..."
2,cb,2,constitucion_de_1917_-_santa_marta,"LINESTRING (-99.06251 19.34486, -99.04345 19.3..."
3,cb,3,los_pinos/constituyentes_-_vasco_de_quiroga,"LINESTRING (-99.2318 19.38422, -99.22905 19.39..."


### TREN_LIGERO

#### Paths

In [1425]:
paradasTL = gpd.read_file("Data/Unzipped/ste_shp/ste_tren_ligero_shp/STE_TrenLigero_estaciones_utm14n.shp")
rutasTL= gpd.read_file("Data/Unzipped/ste_shp/ste_tren_ligero_shp/STE_TrenLigero_linea_utm14n.shp")

#### Stops

In [1426]:
paradasTL

,SISTEMA,NOMBRE,LINEA,EST,CVE_EST,CVE_EOD17,TIPO,ALCALDIAS,AÑO,Ramp_s_rue,Elevadores,Guia_tact,geometry
0,STE Tren ligero,Xochimilco,01,18,TL0118,12016,Terminal,Xochimilco,1988,Sí,No,Sí,POINT Z (488646.343 2129539.511 0)
1,STE Tren ligero,Francisco Goitia,01,17,TL0117,12004,Intermedia,Xochimilco,1988,Sí,No,Sí,POINT Z (488310.092 2129681.621 0)
2,STE Tren ligero,Huichapan,01,16,TL0116,12005,Intermedia,Xochimilco,1988,Sí,No,Sí,POINT Z (487533.543 2130093.345 0)
3,STE Tren ligero,La Noria,01,15,TL0115,12007,Intermedia,Xochimilco,1988,Sí,No,Sí,POINT Z (486806.992 2130477.041 0)
4,STE Tren ligero,Tepepan,01,14,TL0114,12014,Intermedia,Xochimilco,1988,No,No,Sí,POINT Z (486008.543 2131762.838 0)
5,STE Tren ligero,Periférico,01,13,TL0113,12011,Intermedia,Tlalpan,1988,No,No,Sí,POINT Z (485322.453 2132116.775 0)
6,STE Tren ligero,Xomalí,01,12,TL0112,12017,Intermedia,Tlalpan,1988,Sí,No,Sí,POINT Z (484582.228 2132779.859 0)
7,STE Tren ligero,Huipulco,01,11,TL0111,12006,Intermedia,Tlalpan,1988,Sí,No,Sí,POINT Z (484167.964 2133751.946 0)
8,STE Tren ligero,Estadio Azteca,01,10,TL0110,12003,Intermedia,Coyoacán / Tlalpan,1986,No,Sí,Sí,POINT Z (484549.991 2134231.513 0)
9,STE Tren ligero,El Vergel,01,09,TL0109,12002,Intermedia,Coyoacán,1986,No,No,Sí,POINT Z (484970.145 2134825.571 0)


In [1427]:
paradasTLP = paradasTL[["SISTEMA",  "LINEA","NOMBRE", "geometry"]].copy()


In [1428]:
paradasTLP.rename(columns={"SISTEMA": "tipo_transporte", "NOMBRE": "parada", "LINEA": "linea", "geometry": "geom"}, inplace=True)
paradasTLP['tipo_transporte']='TL'
paradasTLP['geom'] = paradasTLP['geom'].apply(lambda geom: transform(to_2d, geom))
paradasTLP = clean_string_columns(paradasTLP)
paradasTLP.sort_values(by=["linea"], inplace=True, ignore_index=True)

In [1429]:
get_duplicates(paradasTLP, subset=["parada", "linea"])

,tipo_transporte,linea,parada,geom


In [1430]:
rows_with_nan_geom(paradasTLP)

,tipo_transporte,linea,parada,geom


#### Routes

In [1431]:
rutasTL

,SISTEMA,LINEA,RUTA,geometry
0,STE Tren ligero,1,Tasqueña - Xochimilco,"LINESTRING Z (488646.343 2129539.511 0, 488604..."


In [1432]:
rutasTLP = rutasTL.copy()
rutasTLP.rename(columns={"SISTEMA": "tipo_transporte", "LINEA": "linea", "RUTA": "ruta", "geometry": "geom"}, inplace=True)
rutasTLP['tipo_transporte']='TL'
rutasTLP['geom'] = rutasTLP['geom'].apply(lambda geom: transform(to_2d, geom))
rutasTLP = clean_string_columns(rutasTLP)

#### Finals

In [1433]:
paradasTLP

,tipo_transporte,linea,parada,geom
0,tl,01,xochimilco,POINT (488646.343 2129539.511)
1,tl,01,ciudad_jardin,POINT (485099.322 2137982.162)
2,tl,01,la_virgen,POINT (485229.439 2137536.696)
3,tl,01,xotepingo,POINT (485366.352 2137070.253)
4,tl,01,nezahualpilli,POINT (485486.974 2136660.984)
5,tl,01,registro_federal,POINT (485419.864 2136014.353)
6,tl,01,textitlan,POINT (485231.777 2135418.948)
7,tl,01,el_vergel,POINT (484970.145 2134825.571)
8,tl,01,estadio_azteca,POINT (484549.991 2134231.513)
9,tl,01,huipulco,POINT (484167.964 2133751.946)


In [1434]:
rutasTLP

,tipo_transporte,linea,ruta,geom
0,tl,1,tasquena_-_xochimilco,"LINESTRING (488646.343 2129539.511, 488604.736..."


### TROLEBUS

#### Paths

In [1435]:
paradasTLB = gpd.read_file("Data/Unzipped/ste_shp/ste_trolebus_shp/STE_Trolebus_Paradas.shp")
rutasTLB = gpd.read_file("Data/Unzipped/ste_shp/ste_trolebus_shp/STE_Trolebus_Lineas.shp")

#### Stops

In [1436]:
paradasTLB

,CVE_EST,circuito,SISTEMA,NOMBRE,TIPO,LINEA,ALCALDIAS,Ramp_s_rue,geometry
0,TROLE05R55,1 y 2,STE Trolebús,T. San Felipe de Jesús,Terminal,05,Gustavo A. Madero,None,POINT Z (-99.06508 19.49606 0)
1,TROLE05I49,1 y 2,STE Trolebús,Tres Culturas,Intermedia,05,Cuauhtémoc,None,POINT Z (-99.13276 19.45091 0)
2,TROLE05I50,1 y 2,STE Trolebús,Glorieta Cuitláhuac I,Intermedia,05,Cuauhtémoc,None,POINT Z (-99.13391 19.44947 0)
3,TROLE05I51,1 y 2,STE Trolebús,Glorieta Cuitláhuac,Intermedia,05,Cuauhtémoc,None,POINT Z (-99.13576 19.44743 0)
4,TROLE05I53,1 y 2,STE Trolebús,Glorieta Violeta,Intermedia,05,Cuauhtémoc,None,POINT Z (-99.14234 19.44158 0)
...,...,...,...,...,...,...,...,...,...
800,TROLE12R13,None,STE Trolebús,Pacífico,Intermedia,12,Coyoacán,None,POINT Z (-99.14822 19.3353 0)
801,TROLE12R14,None,STE Trolebús,Anillo Circunvalación,Intermedia,12,Coyoacán,None,POINT Z (-99.14725 19.33771 0)
802,TROLE12R15,None,STE Trolebús,Los Pinos,Intermedia,12,Coyoacán,None,POINT Z (-99.14791 19.34003 0)
803,TROLE12R16,None,STE Trolebús,Central,Intermedia,12,Coyoacán,None,POINT Z (-99.14745 19.34228 0)


In [1437]:
paradasTLBP = paradasTLB[["SISTEMA", "LINEA","NOMBRE", "geometry"]].copy()

In [1438]:
paradasTLBP.rename(columns={"SISTEMA": "tipo_transporte", "NOMBRE": "parada", "LINEA": "linea", "geometry": "geom"}, inplace=True)
paradasTLBP['tipo_transporte']='TLB'
paradasTLBP['geom'] = paradasTLBP['geom'].apply(lambda geom: transform(to_2d, geom))
paradasTLBP = clean_string_columns(paradasTLBP)
paradasTLBP.sort_values(by=["linea"], inplace=True, ignore_index=True)


In [1439]:
get_duplicates(paradasTLBP, subset=["parada", "linea"]).sort_values(by=["linea", "parada"], ignore_index=True)

,tipo_transporte,linea,parada,geom
0,tlb,01,ajusco,POINT (-99.15245 19.35865)
1,tlb,01,ajusco,POINT (-99.15179 19.35908)
2,tlb,01,america,POINT (-99.14987 19.34589)
3,tlb,01,america,POINT (-99.14946 19.34523)
4,tlb,01,angel_urraza,POINT (-99.14812 19.37998)
...,...,...,...,...
459,tlb,12,tepalcatzin,POINT (-99.16235 19.32045)
460,tlb,12,tita_avendano,POINT (-99.17468 19.31002)
461,tlb,12,tita_avendano,POINT (-99.174 19.31017)
462,tlb,12,topiltzin,POINT (-99.15978 19.32354)


no dropeare 


In [1440]:
rows_with_nan_geom(paradasTLBP)

,tipo_transporte,linea,parada,geom


#### Routes

In [1441]:
rutasTLB

,circuito,RUTA,SISTEMA,LINEA,geometry
0,None,Perisur - Tasqueña,STE Trolebús,12,"LINESTRING Z (-99.14893 19.34259 0, -99.14817 ..."
1,None,Villa de Cortés - Apatlaco - Tepalcates,STE Trolebús,09,"MULTILINESTRING Z ((-99.134 19.38456 0, -99.13..."
2,1,La Diana - San Felipe de Jesús,STE Trolebús,05,"LINESTRING Z (-99.17063 19.42547 0, -99.16815 ..."
3,2,M. Hidalgo - San Felipe de Jesús,STE Trolebús,05,"LINESTRING Z (-99.14405 19.43705 0, -99.14406 ..."
4,2,San Felipe de Jesús - M. Hidalgo,STE Trolebús,05,"LINESTRING Z (-99.06508 19.49603 0, -99.06553 ..."
5,1,San Felipe de Jesús - La Diana,STE Trolebús,05,"LINESTRING Z (-99.06508 19.49603 0, -99.06553 ..."
6,None,Constitucion de 1917 - Santa Marta,STE Trolebús Elevado,10,"MULTILINESTRING Z ((-99.06414 19.34564 0, -99...."
7,None,Ciudad Universitaria - CETRAM Periférico Oriente,STE Trolebús,07,"LINESTRING Z (-99.19057 19.3355 0, -99.19058 1..."
8,None,Autobuses del Norte - Autobuses del Sur,STE Trolebús,01,"LINESTRING Z (-99.14317 19.42251 0, -99.14357 ..."
9,None,Metro Chapultepec - Metro Pantitlán,STE Trolebús,02,"MULTILINESTRING Z ((-99.12126 19.41373 0, -99...."


In [1442]:
rutasTLBP = rutasTLB[["SISTEMA", "LINEA", "RUTA", "geometry"]].copy()
rutasTLBP.rename(columns={"SISTEMA": "tipo_transporte", "LINEA": "linea", "RUTA": "ruta", "geometry": "geom"}, inplace=True)
rutasTLBP['tipo_transporte']='TLB'
rutasTLBP['geom'] = rutasTLBP['geom'].apply(lambda geom: transform(to_2d, geom))
rutasTLBP = clean_string_columns(rutasTLBP)
rutasTLBP.sort_values(by=["linea"], inplace=True, ignore_index=True)


#### Finals

In [1443]:
paradasTLBP

,tipo_transporte,linea,parada,geom
0,tlb,01,salto_del_agua,POINT (-99.142 19.42739)
1,tlb,01,cumbres_de_acultzingo,POINT (-99.14636 19.39238)
2,tlb,01,luz_savinon,POINT (-99.14686 19.38854)
3,tlb,01,cumbres_de_maltrata,POINT (-99.14715 19.38608)
4,tlb,01,central_del_norte,POINT (-99.14038 19.47942)
...,...,...,...,...
800,tlb,12,los_pinos,POINT (-99.14825 19.34006)
801,tlb,12,division_del_norte,POINT (-99.14883 19.34219)
802,tlb,12,tasquena,POINT (-99.13959 19.34285)
803,tlb,12,perisur,POINT (-99.18595 19.30489)


In [1444]:
rutasTLBP

,tipo_transporte,linea,ruta,geom
0,tlb,01,autobuses_del_norte_-_autobuses_del_sur,"LINESTRING (-99.14317 19.42251, -99.14357 19.4..."
1,tlb,02,metro_chapultepec_-_metro_pantitlan,"MULTILINESTRING ((-99.12126 19.41373, -99.1212..."
2,tlb,03,metro_mixcoac_-_sn._andres_tetepilco,"LINESTRING (-99.1881 19.37583, -99.18787 19.37..."
3,tlb,04,metro_el_rosario_-_metro_boulevard_puerto_aereo,"LINESTRING (-99.19881 19.51131, -99.19855 19.5..."
4,tlb,05,la_diana_-_san_felipe_de_jesus,"LINESTRING (-99.17063 19.42547, -99.16815 19.4..."
5,tlb,05,m._hidalgo_-_san_felipe_de_jesus,"LINESTRING (-99.14405 19.43705, -99.14406 19.4..."
6,tlb,05,san_felipe_de_jesus_-_m._hidalgo,"LINESTRING (-99.06508 19.49603, -99.06553 19.4..."
7,tlb,05,san_felipe_de_jesus_-_la_diana,"LINESTRING (-99.06508 19.49603, -99.06553 19.4..."
8,tlb,06,metro_el_rosario_-_metro_chapultepec,"LINESTRING (-99.19881 19.51131, -99.19855 19.5..."
9,tlb,07,ciudad_universitaria_-_cetram_periferico_oriente,"LINESTRING (-99.19057 19.3355, -99.19058 19.33..."


## Geometry manipulation and standarization

In [1445]:
# Transform all processed GeoDataFrame geometries to EPSG:6372
paradasConceP = paradasConceP.set_geometry("geom").to_crs(epsg=6372)
paradasMBP = paradasMBP.set_geometry("geom").to_crs(epsg=6372)
paradasRTPP = paradasRTPP.set_geometry("geom").to_crs(epsg=6372)
paradasMP = paradasMP.set_geometry("geom").to_crs(epsg=6372)
paradasCBP = paradasCBP.set_geometry("geom").to_crs(epsg=6372)
paradasTLP = paradasTLP.set_geometry("geom").to_crs(epsg=6372)
paradasTLBP = paradasTLBP.set_geometry("geom").to_crs(epsg=6372)

rutasConceP = rutasConceP.set_geometry("geom").to_crs(epsg=6372)
rutasMBP = rutasMBP.set_geometry("geom").to_crs(epsg=6372)
rutasRTPP = rutasRTPP.set_geometry("geom").to_crs(epsg=6372)
rutasMP = rutasMP.set_geometry("geom").to_crs(epsg=6372)
rutasCBP = rutasCBP.set_geometry("geom").to_crs(epsg=6372)
rutasTLP = rutasTLP.set_geometry("geom").to_crs(epsg=6372)
rutasTLBP = rutasTLBP.set_geometry("geom").to_crs(epsg=6372)

## Master paradas table

In [1446]:
paradasConceP

,tipo_transporte,linea,parada,geom
0,cc,10_de_abril_-_centro_de_azcapotzalco,calle_miguel_lerdo_de_tejada_-_calle_santo_dom...,POINT (2793798.941 834518.735)
1,cc,10_de_abril_-_centro_de_azcapotzalco,calle_soberania_estatal_-_calle_gral._lazaro_c...,POINT (2790990.074 835415.642)
2,cc,10_de_abril_-_centro_de_azcapotzalco,c._miguel_velazquez_mancilla_-_calz._de_san_ag...,POINT (2791282.947 835540.567)
3,cc,10_de_abril_-_centro_de_azcapotzalco,calle_venustiano_carranza_-_calle_felipe_angeles,POINT (2791757.914 835387.23)
4,cc,10_de_abril_-_centro_de_azcapotzalco,calle_francisco_sarabia_-_calle_manuel_salazar,POINT (2792220.176 835194.999)
...,...,...,...,...
13079,cc,zacate_-_metro_hidalgo,cda._lago_espiridino_-_av._marina_nacional,POINT (2794488.586 831246.539)
13080,cc,zacate_-_metro_hidalgo,calle_golfo_de_aden_-_cda._lago_espiridino,POINT (2794206.625 831302.46)
13081,cc,zacate_-_metro_hidalgo,calle_lago_gran_oso_-_lago_chiem,POINT (2794094.531 830975.37)
13082,cc,zacate_-_metro_hidalgo,callo_lago_erne_-_calle_lago_kolind,POINT (2793675.508 830789.915)


In [1447]:
paradasMBP

,tipo_transporte,linea,parada,geom
0,mb,01,felix_cuevas,POINT (2795516.518 822523.901)
1,mb,01,colonia_del_valle,POINT (2795879.648 823793.277)
2,mb,01,cd._de_los_deportes,POINT (2795776.915 823434.156)
3,mb,01,parque_hundido,POINT (2795685.925 823116.087)
4,mb,01,rio_churubusco,POINT (2795342.67 821916.198)
...,...,...,...,...
319,mb,07,gustavo_a._madero,POINT (2802067.742 834719.998)
320,mb,07,av._talisman,POINT (2801330.243 834167.285)
321,mb,07,necaxa,POINT (2801207.813 833800.607)
322,mb,07,robles_dominguez,POINT (2800925.4 832954.032)


In [1448]:
paradasRTPP

,tipo_transporte,linea,parada,geom
0,rtp,1-d,avena,POINT (2801835.781 820795.518)
1,rtp,1-d,zacatepec,POINT (2813364.521 820777.089)
2,rtp,1-d,octavio_senties_ii,POINT (2813015.706 820614.252)
3,rtp,1-d,primavera,POINT (2812426.806 820219.32)
4,rtp,1-d,las_palmas_(la_quebradora),POINT (2812185.693 819734.031)
...,...,...,...,...
8127,rtp,9-d,estacionamiento_ibero,POINT (2786726.396 821757.472)
8128,rtp,9-d,ibero,POINT (2786606.2 821626.45)
8129,rtp,9-d,autopista,POINT (2786171.989 821384.781)
8130,rtp,9-d,centro_comercial_santa_fe,POINT (2785739.78 821169.328)


In [1449]:
paradasMP

,tipo_transporte,linea,parada,geom
0,m,01,pantitlan,POINT (2806320.502 827386.15)
1,m,01,observatorio,POINT (2793201.609 825131.389)
2,m,01,tacubaya,POINT (2794529.496 825624.518)
3,m,01,juanacatlan,POINT (2795098.772 826788.015)
4,m,01,chapultepec,POINT (2795664.738 827659.656)
...,...,...,...,...
190,m,b,garibaldi/lagunilla,POINT (2799650.015 830293.589)
191,m,b,guerrero,POINT (2798781.228 830414.841)
192,m,b,plaza_aragon,POINT (2810728.765 839856.37)
193,m,b,oceania,POINT (2804947.854 830599.548)


In [1450]:
paradasCBP

,tipo_transporte,linea,parada,geom
0,cb,01,indios_verdes,POINT (2801169.566 836015.167)
1,cb,01,ticoman,POINT (2799925.623 837781.551)
2,cb,01,la_pastora,POINT (2799159.481 839497.441)
3,cb,01,campos_revolucion,POINT (2799009.568 840919.676)
4,cb,01,cuautepec,POINT (2799786.672 842886.768)
5,cb,01,tlalpexco,POINT (2800624.356 841449.221)
6,cb,02,santa_marta,POINT (2814906.864 821290.936)
7,cb,02,san_miguel_teotongo,POINT (2815408.283 819535.895)
8,cb,02,lomas_de_la_estancia,POINT (2813469.925 819197.428)
9,cb,02,xalpa,POINT (2812339.663 818659.927)


In [1451]:
paradasTLP

,tipo_transporte,linea,parada,geom
0,tl,01,xochimilco,POINT (2803182.478 810001.702)
1,tl,01,ciudad_jardin,POINT (2799469.67 818349.17)
2,tl,01,la_virgen,POINT (2799608.666 817907.562)
3,tl,01,xotepingo,POINT (2799754.877 817445.171)
4,tl,01,nezahualpilli,POINT (2799883.657 817039.465)
5,tl,01,registro_federal,POINT (2799830.107 816393.126)
6,tl,01,textitlan,POINT (2799654.832 815795.372)
7,tl,01,el_vergel,POINT (2799406.157 815198.114)
8,tl,01,estadio_azteca,POINT (2798999.38 814596.895)
9,tl,01,huipulco,POINT (2798628.257 814110.659)


In [1452]:
paradasTLBP

,tipo_transporte,linea,parada,geom
0,tlb,01,salto_del_agua,POINT (2799253.92 828463.885)
1,tlb,01,cumbres_de_acultzingo,POINT (2798874.424 824591.549)
2,tlb,01,luz_savinon,POINT (2798830.463 824166.309)
3,tlb,01,cumbres_de_maltrata,POINT (2798805.07 823894.822)
4,tlb,01,central_del_norte,POINT (2799308.13 834209.428)
...,...,...,...,...
800,tlb,12,los_pinos,POINT (2798790.639 818813.243)
801,tlb,12,division_del_norte,POINT (2798725.198 819047.441)
802,tlb,12,tasquena,POINT (2799691.518 819139.112)
803,tlb,12,perisur,POINT (2794917.099 814854.176)


In [1453]:
paradas_all = pd.concat([
    paradasConceP,
    paradasMBP,
    paradasRTPP,
    paradasMP,
    paradasCBP,
    paradasTLP,
    paradasTLBP
], ignore_index=True)

In [1454]:
paradas_all

,tipo_transporte,linea,parada,geom
0,cc,10_de_abril_-_centro_de_azcapotzalco,calle_miguel_lerdo_de_tejada_-_calle_santo_dom...,POINT (2793798.941 834518.735)
1,cc,10_de_abril_-_centro_de_azcapotzalco,calle_soberania_estatal_-_calle_gral._lazaro_c...,POINT (2790990.074 835415.642)
2,cc,10_de_abril_-_centro_de_azcapotzalco,c._miguel_velazquez_mancilla_-_calz._de_san_ag...,POINT (2791282.947 835540.567)
3,cc,10_de_abril_-_centro_de_azcapotzalco,calle_venustiano_carranza_-_calle_felipe_angeles,POINT (2791757.914 835387.23)
4,cc,10_de_abril_-_centro_de_azcapotzalco,calle_francisco_sarabia_-_calle_manuel_salazar,POINT (2792220.176 835194.999)
...,...,...,...,...
22572,tlb,12,los_pinos,POINT (2798790.639 818813.243)
22573,tlb,12,division_del_norte,POINT (2798725.198 819047.441)
22574,tlb,12,tasquena,POINT (2799691.518 819139.112)
22575,tlb,12,perisur,POINT (2794917.099 814854.176)


## Finals

In [ ]:
import os

# Define transport methods and their processed DataFrames
transport_methods = {
    "cc": {"paradas": paradasConceP, "rutas": rutasConceP},
    "mb": {"paradas": paradasMBP, "rutas": rutasMBP},
    "rtp": {"paradas": paradasRTPP, "rutas": rutasRTPP},
    "m": {"paradas": paradasMP, "rutas": rutasMP},
    "cb": {"paradas": paradasCBP, "rutas": rutasCBP},
    "tl": {"paradas": paradasTLP, "rutas": rutasTLP},
    "tlb": {"paradas": paradasTLBP, "rutas": rutasTLBP},
}

base_dir = "/Data/Finals"
os.makedirs(base_dir, exist_ok=True)

# Save processed DataFrames for each transport method as shapefile
for method, dfs in transport_methods.items():
    method_dir = os.path.join(base_dir, method)
    paradas_dir = os.path.join(method_dir, "paradas")
    rutas_dir = os.path.join(method_dir, "rutas")
    os.makedirs(paradas_dir, exist_ok=True)
    os.makedirs(rutas_dir, exist_ok=True)
    # Save paradas
    dfs["paradas"].to_file(os.path.join(paradas_dir, "paradas.shp"), driver="ESRI Shapefile")
    # Save rutas
    dfs["rutas"].to_file(os.path.join(rutas_dir, "rutas.shp"), driver="ESRI Shapefile")

# Save all_paradas DataFrame as shapefile
all_paradas_dir = os.path.join(base_dir, "all_paradas")
os.makedirs(all_paradas_dir, exist_ok=True)
paradas_all.to_file(os.path.join(all_paradas_dir, "paradas_all.shp"), driver="ESRI Shapefile")

C:\Users\Edu\AppData\Local\Temp\ipykernel_20492\3263090768.py:25: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  dfs["paradas"].to_file(os.path.join(paradas_dir, "paradas.shp"), driver="ESRI Shapefile")
c:\Users\Edu\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'tipo_transporte' to 'tipo_trans'
  ogr_write(
C:\Users\Edu\AppData\Local\Temp\ipykernel_20492\3263090768.py:27: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  dfs["rutas"].to_file(os.path.join(rutas_dir, "rutas.shp"), driver="ESRI Shapefile")
c:\Users\Edu\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'tipo_transporte' to 'tipo_trans'
  ogr_write(
c:\Users\Edu\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'origen_destino' to 'origen_des'
  ogr_write(
C:\Users\Edu\AppData

## Upload

In [ ]:
# Set schema and ensure it exists
schema = "semovi"
ensure_schema(schema)

import os
import geopandas as gpd
#Note ROOT IS 'C:/Users/Edu/OneDrive/1.Data/0.Base', CHECK SQL SETUP PRINT (it automatically sets up for each user)
base_dir = ROOT / "SEMOVI_Edu/Data/Finals"
transport_methods = ["mb", "m", "rtp", "cc", "cb", "tl", "tlb"]

for method in transport_methods:
    paradas_path = base_dir / method / "paradas" / "paradas.shp"
    rutas_path = base_dir / method / "rutas" / "rutas.shp"
    # Table names: MBparadas, MBrutas, etc.
    paradas_table = f"{method.upper()}paradas"
    rutas_table = f"{method.upper()}rutas"
    # Upload paradas
    if paradas_path.exists():
        gdf_paradas = gpd.read_file(paradas_path)
        write_spatial(gdf_paradas, schema, paradas_table)
    # Upload rutas
    if rutas_path.exists():
        gdf_rutas = gpd.read_file(rutas_path)
        write_spatial(gdf_rutas, schema, rutas_table)

# Upload all_paradas
all_paradas_path = base_dir / "all_paradas" / "paradas_all.shp"
if all_paradas_path.exists():
    gdf_all_paradas = gpd.read_file(all_paradas_path)
    write_spatial(gdf_all_paradas, schema, "all_paradas")